In [ ]:
# Step 1: Install Dependencies

!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.5 MB/s eta 0:00:00


In [ ]:
# Step 2: Load Dataset (Pipeline 1)

from datasets import load_dataset, DatasetDict

dataset = load_dataset("zeroshot/twitter-financial-news-sentiment")
print(dataset)
print(dataset["train"][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sent_train.csv: 0.00B [00:00, ?B/s]

sent_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/9543 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2388 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})
{'text': '$BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT', 'label': 0}


In [ ]:
# Step 3: Sample Dataset (Pipeline 1)

# Sample 5000 from train, split validation in half
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(5000))
validation_full = dataset["validation"].shuffle(seed=42)

half = len(validation_full) // 2
dataset["validation"] = validation_full.select(range(half))
dataset["test"]       = validation_full.select(range(half, len(validation_full)))

print(f"Train:      {len(dataset['train'])}")
print(f"Validation: {len(dataset['validation'])}")
print(f"Test:       {len(dataset['test'])}")

Train:      5000
Validation: 1194
Test:       1194


In [ ]:
# Step 4: Load Tokenizer and Process Data (Pipeline 1)

from transformers import AutoTokenizer, DataCollatorWithPadding

model_name = "jtatman/finetuning-twitter-finance-sentiment-distilbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    # Use dynamic padding instead of static padding
    return tokenizer(batch["text"], truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(tokenized_dataset)

config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/322 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1194 [00:00<?, ? examples/s]

Map:   0%|          | 0/1194 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1194
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1194
    })
})


In [ ]:
# Step 5: Load Pre-trained Model (Pipeline 1)

from transformers import AutoModelForSequenceClassification

# Define label mappings (3 sentiment classes)
id2label = {0: "Bearish", 1: "Bullish", 2: "Neutral"}
label2id = {"Bearish": 0, "Bullish": 1, "Neutral": 2}
num_labels = 3

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

print("Labels:", id2label)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Labels: {0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}


In [ ]:
# Step 6: Define Evaluation Metrics (Pipeline 1)

import evaluate
import numpy as np

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [ ]:
# Step 7: Training Configuration (Pipeline 1)

from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finbert-sentiment-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir="./logs",
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    data_collator=data_collator,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# Step 8: Train Model (Pipeline 1)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.545058,0.873534


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=157, training_loss=0.16264900887847705, metrics={'train_runtime': 32.3824, 'train_samples_per_second': 154.405, 'train_steps_per_second': 4.848, 'total_flos': 70898230068048.0, 'train_loss': 0.16264900887847705, 'epoch': 1.0})

In [ ]:
# Step 9: Evaluate on Test Set (Pipeline 1)

import time

start = time.time()
results = trainer.evaluate(tokenized_dataset["test"])
end = time.time()

print(f"Test Accuracy: {results['eval_accuracy']:.4f}")
print(f"Runtime:       {end - start:.2f} seconds")

Test Accuracy: 0.8819
Runtime:       1.76 seconds


In [ ]:
# Step 10: Upload Pipeline 1 Model to HuggingFace

from huggingface_hub import login
from transformers import AutoModelForSequenceClassification, AutoTokenizer

login()

repo_name_sent = "Nora0211/finbert-sentiment-finetuned"

# Save model and tokenizer locally
trainer.save_model("./finbert-sentiment-finetuned")
tokenizer.save_pretrained("./finbert-sentiment-finetuned")

# Reload and upload to HuggingFace
model_sent    = AutoModelForSequenceClassification.from_pretrained("./finbert-sentiment-finetuned")
tokenizer_sent = AutoTokenizer.from_pretrained("./finbert-sentiment-finetuned")

model_sent.push_to_hub(repo_name_sent)
tokenizer_sent.push_to_hub(repo_name_sent)

print("Pipeline 1 model uploaded successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...8ojhm64/model.safetensors:  18%|#7        | 48.0MB /  268MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Pipeline 1 model uploaded successfully.


In [ ]:
# Step 11: Load Dataset (Pipeline 2)

dataset2 = load_dataset("zeroshot/twitter-financial-news-topic")

# Sample 5000 from train, split 10% as validation; use original validation as test
train_full2 = dataset2["train"].shuffle(seed=42).select(range(5000))
split2 = train_full2.train_test_split(test_size=0.1, seed=42)

dataset2 = DatasetDict({
    "train":      split2["train"],
    "validation": split2["test"],
    "test":       dataset2["validation"].shuffle(seed=42).select(range(1000))
})

print(f"Train:      {len(dataset2['train'])}")
print(f"Validation: {len(dataset2['validation'])}")
print(f"Test:       {len(dataset2['test'])}")

README.md: 0.00B [00:00, ?B/s]

topic_train.csv: 0.00B [00:00, ?B/s]

topic_valid.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/16990 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4117 [00:00<?, ? examples/s]

Train:      4500
Validation: 500
Test:       1000


In [ ]:
# Step 12: Define Label Mappings (Pipeline 2)

# 20 financial topic classes
id2label2 = {
    0:  "Analyst Update",
    1:  "Fed & Central Banks",
    2:  "Company & Product News",
    3:  "Treasuries & Corporate Debt",
    4:  "Dividend",
    5:  "Earnings",
    6:  "Energy & Oil",
    7:  "Financials",
    8:  "Currencies",
    9:  "General News & Opinion",
    10: "Gold, Metals & Materials",
    11: "IPO",
    12: "Legal & Regulation",
    13: "M&A & Investments",
    14: "Macro & Economy",
    15: "Markets",
    16: "Politics",
    17: "Personnel Change",
    18: "Stock Commentary",
    19: "Stock Movement"
}

label2id2  = {v: k for k, v in id2label2.items()}
num_labels2 = 20
print("Number of labels:", num_labels2)

Number of labels: 20


In [ ]:
# Step 13: Load Tokenizer and Process Data (Pipeline 2)

model_name2 = "leonas5555/finnews-topic-single-classify"
tokenizer2  = AutoTokenizer.from_pretrained(model_name2)

def tokenize2(batch):
    return tokenizer2(batch["text"], truncation=True, max_length=128)

tokenized_dataset2 = dataset2.map(tokenize2, batched=True)
data_collator2     = DataCollatorWithPadding(tokenizer=tokenizer2)

print(tokenized_dataset2)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4500
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 500
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1000
    })
})


In [ ]:
# Step 14: Load Pre-trained Model (Pipeline 2)

model2 = AutoModelForSequenceClassification.from_pretrained(
    model_name2,
    num_labels=num_labels2,
    id2label=id2label2,
    label2id=label2id2,
    ignore_mismatched_sizes=True
)


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
# Step 15: Define Evaluation Metrics (Pipeline 2)

import evaluate

metric2 = evaluate.load("accuracy")

def compute_metrics2(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric2.compute(predictions=predictions, references=labels)

In [ ]:
# Step 16: Training Configuration (Pipeline 2)

training_args2 = TrainingArguments(
    output_dir="./finbert-topic-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    logging_dir="./logs2",
    report_to="none"
)

trainer2 = Trainer(
    model=model2,
    args=training_args2,
    train_dataset=tokenized_dataset2["train"],
    eval_dataset=tokenized_dataset2["validation"],
    compute_metrics=compute_metrics2,
    data_collator=data_collator2,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
# Step 17: Train Model (Pipeline 2)

trainer2.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.016993,0.996000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=141, training_loss=0.028854654190388133, metrics={'train_runtime': 28.8929, 'train_samples_per_second': 155.748, 'train_steps_per_second': 4.88, 'total_flos': 188257602624960.0, 'train_loss': 0.028854654190388133, 'epoch': 1.0})

In [ ]:
# Step 18: Evaluate on Test Set (Pipeline 2)

start2 = time.time()
results2 = trainer2.evaluate(tokenized_dataset2["test"])
end2 = time.time()

print(f"Test Accuracy: {results2['eval_accuracy']:.4f}")
print(f"Runtime:       {end2 - start2:.2f} seconds")

Test Accuracy: 0.8830
Runtime:       1.38 seconds


In [ ]:
# Step 19: Upload Pipeline 2 Model to HuggingFace

login()

repo_name_topic = "Nora0211/finbert-topic-finetuned"

# Save model and tokenizer locally
trainer2.save_model("./finbert-topic-finetuned")
tokenizer2.save_pretrained("./finbert-topic-finetuned")

# Reload and upload to HuggingFace
model_topic    = AutoModelForSequenceClassification.from_pretrained("./finbert-topic-finetuned")
tokenizer_topic = AutoTokenizer.from_pretrained("./finbert-topic-finetuned")

model_topic.push_to_hub(repo_name_topic)
tokenizer_topic.push_to_hub(repo_name_topic)

print("Pipeline 2 model uploaded successfully.")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...3xrjlyi/model.safetensors:  25%|##5       |  112MB /  439MB            

No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


Pipeline 2 model uploaded successfully.


In [ ]:
# Step 20: Verify Both Models on HuggingFace

from huggingface_hub import model_info

for repo in [repo_name_sent, repo_name_topic]:
    try:
        info = model_info(repo)
        print(f"✅ Model exists: {repo}")
        print(f"   Files: {[f.rfilename for f in info.siblings]}")
    except Exception as e:
        print(f"❌ Model not found: {repo} → {e}")

✅ Model exists: Nora0211/finbert-sentiment-finetuned
   Files: ['.gitattributes', 'README.md', 'config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
✅ Model exists: Nora0211/finbert-topic-finetuned
   Files: ['.gitattributes', 'README.md', 'config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']
